# E-Commerce Sales Analytics — Python Analysis

This notebook combines **Data Understanding, Data Cleaning, Feature Engineering, and Exploratory Data Analysis (EDA)** into one simple workflow.

**Tools:** Python, Pandas, NumPy, Matplotlib

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


## 2. Load the Dataset

Expected project structure:

```text
E-Commerce-Sales-Analytics/
├── Data/
│   └── realistic_e_commerce_sales_data.csv
└── Python/
    └── E_Commerce_Sales_Analytics_Complete.ipynb
```

In [ ]:
DATA_PATH = Path("../Data/realistic_e_commerce_sales_data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}. "
        "Place realistic_e_commerce_sales_data.csv inside the Data folder."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


## 3. Data Understanding

In [ ]:
display(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.info()


In [ ]:
print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False))
print("Duplicate rows:", df.duplicated().sum())


In [ ]:
print("Gender:", df["Gender"].dropna().unique())
print("Region:", df["Region"].dropna().unique())
print("Category:", df["Category"].dropna().unique())
print("Shipping Status:", df["Shipping Status"].dropna().unique())
print("Unique Products:", df["Product Name"].nunique())
print("Unique Customers:", df["Customer ID"].nunique())


## 4. Data Cleaning

In [ ]:
text_columns = [
    "Customer ID", "Gender", "Region",
    "Product Name", "Category", "Shipping Status"
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Region"] = df["Region"].fillna("Unknown")
df["Shipping Status"] = df["Shipping Status"].fillna("Unknown")
df["Age"] = df["Age"].fillna(df["Age"].median())

df = df.drop_duplicates().reset_index(drop=True)

print("Cleaning completed.")
print("Rows after cleaning:", len(df))


In [ ]:
print("Invalid dates:", df["Order Date"].isna().sum())
print("Negative unit prices:", (df["Unit Price"] < 0).sum())
print("Invalid quantities:", (df["Quantity"] <= 0).sum())
print("Negative total prices:", (df["Total Price"] < 0).sum())
print("Negative shipping fees:", (df["Shipping Fee"] < 0).sum())

missing = df.isnull().sum()
display(missing[missing > 0])


## 5. Price Validation

In [ ]:
df["Calculated Total"] = df["Unit Price"] * df["Quantity"]
df["Difference"] = df["Total Price"] - df["Calculated Total"]

df["Price Check"] = np.where(
    np.isclose(df["Total Price"], df["Calculated Total"]),
    "Valid",
    "Review"
)

display(df["Price Check"].value_counts())
print(f"Records needing price review: {df['Price Check'].eq('Review').mean() * 100:.2f}%")

df.drop(columns=["Calculated Total"], inplace=True)


## 6. Feature Engineering

In [ ]:
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month Name"] = df["Order Date"].dt.month_name()
df["Quarter"] = "Q" + df["Order Date"].dt.quarter.astype(str)
df["Year Month"] = df["Order Date"].dt.to_period("M").astype(str)

age_bins = [0, 25, 35, 45, 55, np.inf]
age_labels = ["18-25", "26-35", "36-45", "46-55", "56+"]

df["Age Group"] = pd.cut(
    df["Age"], bins=age_bins, labels=age_labels, include_lowest=True
)

df["Delivered Flag"] = df["Shipping Status"].eq("Delivered").astype(int)
df["Returned Flag"] = df["Shipping Status"].eq("Returned").astype(int)
df["In Transit Flag"] = df["Shipping Status"].eq("In Transit").astype(int)

df["Revenue"] = df["Total Price"]

df["Order Value Segment"] = pd.cut(
    df["Revenue"],
    bins=[0, 500, 1000, 2500, np.inf],
    labels=["Low", "Medium", "High", "Very High"],
    include_lowest=True
)

print("Feature engineering completed.")


## 7. Business KPIs

In [ ]:
total_revenue = df["Revenue"].sum()
total_orders = len(df)
total_quantity = df["Quantity"].sum()
unique_customers = df["Customer ID"].nunique()
average_order_value = total_revenue / total_orders if total_orders else 0
total_returns = df["Returned Flag"].sum()
return_rate = total_returns / total_orders * 100 if total_orders else 0

final_kpis = pd.DataFrame({
    "Metric": [
        "Total Revenue", "Total Orders", "Total Quantity",
        "Unique Customers", "Average Order Value",
        "Total Returns", "Return Rate"
    ],
    "Value": [
        total_revenue, total_orders, total_quantity,
        unique_customers, round(average_order_value, 2),
        total_returns, round(return_rate, 2)
    ]
})

display(final_kpis)


## 8. Product Analysis

In [ ]:
product_summary = (
    df.groupby("Product Name")
    .agg(
        Revenue=("Revenue", "sum"),
        Quantity=("Quantity", "sum"),
        Orders=("Product Name", "count"),
        Average_Price=("Unit Price", "mean")
    )
    .sort_values("Revenue", ascending=False)
)

display(product_summary.head(10))


In [ ]:
top_products = product_summary.head(10)

plt.figure(figsize=(10, 5))
plt.bar(top_products.index.astype(str), top_products["Revenue"])
plt.title("Top 10 Products by Revenue")
plt.xlabel("Product")
plt.ylabel("Revenue")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9. Category Analysis

In [ ]:
category_summary = (
    df.groupby("Category")
    .agg(
        Revenue=("Revenue", "sum"),
        Quantity=("Quantity", "sum"),
        Orders=("Category", "count"),
        Average_Order_Value=("Revenue", "mean")
    )
    .sort_values("Revenue", ascending=False)
)

category_summary["Revenue %"] = (
    category_summary["Revenue"] / category_summary["Revenue"].sum() * 100
)

display(category_summary)


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(category_summary.index.astype(str), category_summary["Revenue"])
plt.title("Revenue by Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()


## 10. Region Analysis

In [ ]:
region_summary = (
    df.groupby("Region")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Region", "count"),
        Quantity=("Quantity", "sum"),
        Average_Order_Value=("Revenue", "mean")
    )
    .sort_values("Revenue", ascending=False)
)

region_summary["Revenue %"] = (
    region_summary["Revenue"] / region_summary["Revenue"].sum() * 100
)

display(region_summary)


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(region_summary.index.astype(str), region_summary["Revenue"])
plt.title("Revenue by Region")
plt.xlabel("Region")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()


## 11. Monthly Revenue Trend

In [ ]:
monthly_summary = (
    df.groupby("Year Month")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Customer ID", "count"),
        Quantity=("Quantity", "sum")
    )
    .sort_index()
)

monthly_summary["Average Order Value"] = (
    monthly_summary["Revenue"] / monthly_summary["Orders"]
)

display(monthly_summary)


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly_summary.index, monthly_summary["Revenue"], marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Highest Revenue Month:", monthly_summary["Revenue"].idxmax())
print("Lowest Revenue Month:", monthly_summary["Revenue"].idxmin())


## 12. Shipping Analysis

In [ ]:
shipping_summary = (
    df["Shipping Status"]
    .value_counts()
    .rename_axis("Shipping Status")
    .reset_index(name="Orders")
)

shipping_summary["Percentage"] = (
    shipping_summary["Orders"] / shipping_summary["Orders"].sum() * 100
)

display(shipping_summary)


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(shipping_summary["Shipping Status"], shipping_summary["Orders"])
plt.title("Orders by Shipping Status")
plt.xlabel("Shipping Status")
plt.ylabel("Orders")
plt.tight_layout()
plt.show()


## 13. Return Analysis

In [ ]:
returned_revenue = df.loc[df["Returned Flag"].eq(1), "Revenue"].sum()

returns_by_product = (
    df.loc[df["Returned Flag"].eq(1)]
    .groupby("Product Name").size()
    .sort_values(ascending=False)
)

returns_by_category = (
    df.loc[df["Returned Flag"].eq(1)]
    .groupby("Category").size()
    .sort_values(ascending=False)
)

returns_by_region = (
    df.loc[df["Returned Flag"].eq(1)]
    .groupby("Region").size()
    .sort_values(ascending=False)
)

print(f"Total Returns: {total_returns}")
print(f"Return Rate: {return_rate:.2f}%")
print(f"Revenue from Returned Orders: {returned_revenue:,.2f}")

print("\nTop Returned Products:")
display(returns_by_product.head(10))
print("\nReturns by Category:")
display(returns_by_category)
print("\nReturns by Region:")
display(returns_by_region)


In [ ]:
category_return_rate = (
    df.groupby("Category")["Returned Flag"]
    .mean().mul(100).sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
plt.bar(category_return_rate.index.astype(str), category_return_rate.values)
plt.title("Return Rate by Category")
plt.xlabel("Category")
plt.ylabel("Return Rate (%)")
plt.tight_layout()
plt.show()


## 14. Customer Analysis

In [ ]:
customer_summary = (
    df.groupby("Customer ID")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Customer ID", "count"),
        Quantity=("Quantity", "sum"),
        Average_Order_Value=("Revenue", "mean")
    )
    .sort_values("Revenue", ascending=False)
)

customer_summary["Revenue %"] = (
    customer_summary["Revenue"] / customer_summary["Revenue"].sum() * 100
)

display(customer_summary.head(10))


## 15. Gender and Age Group Analysis

In [ ]:
gender_analysis = (
    df.groupby("Gender")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Gender", "count"),
        Average_Order_Value=("Revenue", "mean")
    )
    .sort_values("Revenue", ascending=False)
)

age_group_analysis = (
    df.groupby("Age Group", observed=True)
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Age Group", "count"),
        Average_Order_Value=("Revenue", "mean")
    )
    .sort_values("Revenue", ascending=False)
)

print("Gender Analysis")
display(gender_analysis)

print("Age Group Analysis")
display(age_group_analysis)


## 16. Final Data Quality Check

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate Rows:", df.duplicated().sum())
print("Total Missing Values:", int(df.isnull().sum().sum()))
print("Price Records for Review:", int(df["Price Check"].eq("Review").sum()))


## 17. Export Cleaned Dataset

In [ ]:
output_dir = Path("../Data/Cleaned")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "ecommerce_sales_final.csv"
df.to_csv(output_file, index=False)

print(f"Cleaned dataset saved to: {output_file.resolve()}")


## Conclusion

This single notebook combines the original three notebooks into one clean workflow:

1. Data Understanding
2. Data Cleaning
3. Price Validation
4. Feature Engineering
5. KPI Analysis
6. Product Analysis
7. Category Analysis
8. Region Analysis
9. Monthly Revenue Analysis
10. Shipping Analysis
11. Return Analysis
12. Customer Analysis
13. Gender and Age Group Analysis
14. Final Data Quality Check

The cleaned dataset can be used as the source for the Power BI dashboard.
